# Acción Comercial: Estrategia de Retención

**Objetivo:** Diseñar un plan de acciones comerciales para **10.000 nuevos clientes** que maximice el retorno sobre la inversión (ROI) minimizando la fuga (churn).

El análisis se estructura en tres bloques:
1. **Predicción de churn** — modelo XGBoost entrenado sobre el histórico de clientes
2. **Segmentación y acciones** — estrategia basada únicamente en riesgo de churn (5 segmentos)
3. **Estrategia avanzada** — segmentación 2D combinando riesgo × Customer Lifetime Value (CLTV)

---

## Fórmulas del modelo

### Coste de revisión
$$C(n) = \text{BASE} \times (1 + \alpha)^n$$
- **BASE** = `Mantenimiento_medio` según el modelo del vehículo (tabla de costes)  
- $\alpha$ = 7 % para modelos A y B · 10 % para modelos C–K  
- $n$ = número de revisión futura ($n = \text{revisiones actuales} + k$)

### Margen bruto
$$\text{Margen\_bruto} = C(n) \times 0{,}62 \qquad \text{(costes fijos = 38\,\%)}$$

### Customer Lifetime Value (CLTV)
$$\text{CLTV} = \sum_{k=1}^{N} C(n+k) \times 0{,}62 \times (1 - p_{\text{churn}})^k$$
Clientes con mayor $p_{\text{churn}}$ tienen menor CLTV: es probable que abandonen antes de las revisiones más rentables.

### ROI de la campaña
$$\text{ROI} = \frac{\text{Margen\_bruto} - \text{Costes\_campaña}}{\text{Costes\_campaña}}$$
donde $\text{Costes\_campaña}$ = contacto + coste adicional + marketing + bono (sin incluir el coste fijo del taller).

## 1. Carga y Preparación de Datos

Cargamos tres fuentes de datos:
- **`datamart_final_v2.csv`** — histórico de clientes para entrenar el modelo
- **`nuevos_clientes.csv`** — los 10.000 clientes sobre los que aplicaremos la estrategia
- **`Costes.csv`** — BASE de mantenimiento y margen por modelo de vehículo

In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:,.2f}'.format

# ── Cargar datos ─────────────────────────────────────────────
dm       = pd.read_csv('Data/Datamart/datamart_final_v2.csv')
df_raw   = pd.read_csv('Data/DataLake/customer_data.csv')
df_fresh = pd.read_csv('Data/DataLake/nuevos_clientes.csv')
costes   = pd.read_csv('Data/DataLake/Costes.csv')

print(f'Datos de entrenamiento: {len(dm):,} clientes')
print(f'Nuevos clientes:        {len(df_fresh):,} clientes')
print(f'\nCostes por modelo:')
print(costes[['Modelo', 'Mantenimiento_medio', 'Margen']].to_string(index=False))

Datos de entrenamiento: 58,049 clientes
Nuevos clientes:        10,000 clientes

Costes por modelo:
Modelo  Mantenimiento_medio  Margen
     A               250.00   28.00
     B               263.00   33.00
     C               276.00   33.00
     D               290.00   33.00
     E               305.00   37.00
     F               320.00   42.00
     G               336.00   42.00
     H               353.00   42.00
     I               371.00   43.00
     J               390.00    5.00
     K               410.00    5.00


In [7]:
# ── Construir Churn_Final (Enfoque 2) ────────────────────────
df_raw['Sales_Date'] = pd.to_datetime(df_raw['Sales_Date'], format='%d/%m/%Y', errors='coerce')
fecha_corte = pd.to_datetime('31/12/2023', format='%d/%m/%Y')
df_raw['antiguedad_dias'] = (fecha_corte - df_raw['Sales_Date']).dt.days.clip(lower=0)
df_raw['Churn_bin'] = (df_raw['Churn_400'] == 'Y').astype(int)

dm['Churn_Final'] = (
    (df_raw['Churn_bin'].values == 1) |
    ((df_raw['Revisiones'].values == 0) & (df_raw['antiguedad_dias'].values > 400))
).astype(int)

print(f"Tasa de churn: {dm['Churn_Final'].mean()*100:.1f}%")

Tasa de churn: 33.3%


## 2. Entrenamiento XGBoost y Predicción

Entrenamos un modelo XGBoost sobre el histórico de clientes para obtener la **probabilidad de churn** de cada uno de los 10.000 nuevos clientes.

El output de esta sección es el vector `probs`: una probabilidad entre 0 y 1 por cliente, que usaremos para segmentar y calcular el CLTV.

In [8]:
# ── Features ────────────────────────────────────────────────
FEATURES_NUMERICAS = [
    'km_ultima_revision', 'PVP', 'Edad', 'RENTA_MEDIA_ESTIMADA',
    'gasto_relativo', 'Kw', 'Margen_eur_bruto', 'Margen_eur',
    'ENCUESTA_CLIENTE_ZONA_TALLER'
]
FEATURES_BINARIAS = [
    'tiene_queja', 'en_garantia_bin', 'MANTENIMIENTO_GRATUITO',
    'Lead_compra', 'seguro_bateria_bin', 'sin_encuesta', 'origen_internet'
]
FEATURES_CATEGORICAS = [
    'Modelo', 'ZONA', 'FORMA_PAGO', 'TIPO_CARROCERIA',
    'Equipamiento', 'Fuel', 'TRANSMISION_ID'
]
ALL_FEATURES = FEATURES_NUMERICAS + FEATURES_BINARIAS + FEATURES_CATEGORICAS
COLS_DROP = ['Churn_bin', 'Churn_Final', 'Churn_Corregido', 'perfil_cliente']

# ── Entrenamiento ────────────────────────────────────────────
X = dm.drop(columns=[c for c in COLS_DROP if c in dm.columns])
y = dm['Churn_Final']

label_encoders = {}
for col in FEATURES_CATEGORICAS:
    le = LabelEncoder()
    le.fit(df_raw[col].astype(str))
    label_encoders[col] = le

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
ratio = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=200, max_depth=3, min_child_weight=10,
    gamma=1.0, reg_alpha=1.0, reg_lambda=5.0,
    learning_rate=0.05, scale_pos_weight=ratio,
    random_state=42, eval_metric='logloss', n_jobs=-1
)
xgb_model.fit(X_train, y_train)

# ── Preprocesar nuevos clientes y predecir ───────────────────
fresh = df_fresh.copy()
fresh['QUEJA'] = fresh['QUEJA'].fillna('NO')
fresh['tiene_queja']       = (fresh['QUEJA'].str.upper().str.strip() == 'SI').astype(int)
fresh['en_garantia_bin']   = (fresh['EN_GARANTIA'].fillna('NO').str.upper().str.strip() == 'SI').astype(int)
fresh['gasto_relativo']    = fresh['PVP'] / fresh['RENTA_MEDIA_ESTIMADA'].clip(lower=1)
fresh['seguro_bateria_bin']= (fresh['SEGURO_BATERIA_LARGO_PLAZO'].fillna('NO').str.upper().str.strip() == 'SI').astype(int)
fresh['sin_encuesta']      = fresh['ENCUESTA_CLIENTE_ZONA_TALLER'].isnull().astype(int)
fresh['ENCUESTA_CLIENTE_ZONA_TALLER'] = fresh['ENCUESTA_CLIENTE_ZONA_TALLER'].fillna(0)
fresh['origen_internet']   = (fresh['Origen'].fillna('Tienda').str.strip() == 'Internet').astype(int)

fresh_feat = fresh[ALL_FEATURES].copy()
for col in FEATURES_NUMERICAS:
    if col != 'ENCUESTA_CLIENTE_ZONA_TALLER' and fresh_feat[col].isnull().sum() > 0:
        fresh_feat[col] = fresh_feat[col].fillna(fresh_feat[col].median())
for col in FEATURES_CATEGORICAS:
    if fresh_feat[col].isnull().sum() > 0:
        fresh_feat[col] = fresh_feat[col].fillna(fresh_feat[col].mode()[0])
    le = label_encoders[col]
    known = set(le.classes_)
    fresh_feat[col] = fresh_feat[col].astype(str).apply(
        lambda x, k=known, enc=le: enc.transform([x])[0] if x in k else -1
    )

fresh_aligned = fresh_feat.reindex(columns=X_train.columns, fill_value=0)
probs = xgb_model.predict_proba(fresh_aligned)[:, 1]

print(f'Modelo entrenado | {len(dm):,} clientes de histórico')
print(f'Nuevos clientes  | Media p(churn): {probs.mean():.2%}  ')
print(f'                 | Min: {probs.min():.2%}   Max: {probs.max():.2%}')

Modelo entrenado | 58,049 clientes de histórico
Nuevos clientes  | Media p(churn): 53.19%  
                 | Min: 0.71%   Max: 82.99%


## 3. Segmentación en 5 Grupos de Riesgo

Clasificamos cada cliente en uno de cinco segmentos según su probabilidad de churn:

| Segmento | Umbral | Interpretación |
|----------|--------|----------------|
| MUY_BAJO | < 20 % | Cliente fiel, muy improbable que abandone |
| BAJO     | 20–40 % | Riesgo moderado-bajo |
| MEDIO    | 40–60 % | Riesgo intermedio, requiere seguimiento |
| ALTO     | 60–80 % | Riesgo elevado, acción proactiva necesaria |
| MUY_ALTO | ≥ 80 % | Riesgo crítico, retención urgente |

In [9]:
# ── Segmentación ──────────────────────────────────────────────
def segmento_riesgo(prob):
    if prob >= 0.80: return 'MUY_ALTO'
    if prob >= 0.60: return 'ALTO'
    if prob >= 0.40: return 'MEDIO'
    if prob >= 0.20: return 'BAJO'
    return 'MUY_BAJO'

clientes = df_fresh[['CODE', 'Customer_ID', 'Modelo', 'ZONA', 'PVP', 'Edad', 'Revisiones']].copy()
clientes['prob_churn'] = probs
clientes['segmento'] = clientes['prob_churn'].apply(segmento_riesgo)

print("\nDistribución por segmento:")
for seg in ['MUY_BAJO', 'BAJO', 'MEDIO', 'ALTO', 'MUY_ALTO']:
    n = (clientes['segmento'] == seg).sum()
    if n > 0:
        prob_media = clientes[clientes['segmento'] == seg]['prob_churn'].mean()
        print(f"  {seg:10s}: {n:5,} clientes ({n/len(clientes)*100:5.1f}%) - Prob media: {prob_media:.2%}")


Distribución por segmento:
  MUY_BAJO  : 1,450 clientes ( 14.5%) - Prob media: 6.80%
  BAJO      :    31 clientes (  0.3%) - Prob media: 27.21%
  MEDIO     : 3,916 clientes ( 39.2%) - Prob media: 53.84%
  ALTO      : 4,343 clientes ( 43.4%) - Prob media: 66.60%
  MUY_ALTO  :   260 clientes (  2.6%) - Prob media: 81.12%


## 4. Cálculo de Costes y Beneficios

Para cada cliente calculamos:
- **`C_n`**: coste de la próxima revisión usando la fórmula $C(n) = \text{BASE} \times (1+\alpha)^{n+1}$
- **`Margen_Bruto`**: ingreso neto para el taller = $C(n) \times 0{,}62$
- **`Coste_Marketing`**: 1 % del coste de revisión
- **`Descuento_€`**: bono fijo asignado según el segmento de riesgo

Estas variables alimentan directamente el cálculo del ROI en la sección siguiente.

In [10]:
# ── Fusionar con Costes.csv ──────────────────────────────────
clientes = clientes.merge(costes[['Modelo', 'Mantenimiento_medio']], on='Modelo', how='left')

# ── Fórmula del Profesor ─────────────────────────────────────
# C(n) = BASE × (1 + α)^n
#   BASE = Mantenimiento_medio
#   α    = 7% para modelos A y B, 10% para el resto
#   n    = número de la PRÓXIMA revisión = Revisiones + 1
#
# Costes fijos totales = 38% del ingreso  (1% mkt + 30% CF + 7% comisión)
# Beneficio neto       = C(n) × 0.62

def calcular_ingreso(row):
    base  = row['Mantenimiento_medio']
    alpha = 0.07 if row['Modelo'] in ['A', 'B'] else 0.10
    n     = row['Revisiones'] + 1
    return base * ((1 + alpha) ** n)

clientes['C_n']             = clientes.apply(calcular_ingreso, axis=1)
clientes['Margen_Bruto']    = clientes['C_n'] * 0.62
clientes['Coste_Marketing'] = clientes['C_n'] * 0.01

resumen = clientes.groupby('Modelo').agg(
    BASE            = ('Mantenimiento_medio', 'first'),
    N               = ('C_n', 'count'),
    C_n_medio       = ('C_n', 'mean'),
    Margen_Bruto    = ('Margen_Bruto', 'mean'),
    Coste_Marketing = ('Coste_Marketing', 'mean')
).round(2)
print(resumen.to_string())
print(f"\nIngreso medio por revisión: {clientes['C_n'].mean():,.0f} €")
print(f"Beneficio neto medio:       {clientes['Margen_Bruto'].mean():,.0f} €")

         BASE     N  C_n_medio  Margen_Bruto  Coste_Marketing
Modelo                                                       
A      250.00  1277     267.50        165.85             2.68
B      263.00  2779     281.41        174.47             2.81
C      276.00   705     303.60        188.23             3.04
D      290.00  1547     319.00        197.78             3.19
E      305.00   118     335.50        208.01             3.36
F      320.00     2     352.00        218.24             3.52
G      336.00     7     369.60        229.15             3.70
H      353.00   473     388.30        240.75             3.88
I      371.00  2625     408.10        253.02             4.08
J      390.00   451     429.00        265.98             4.29
K      410.00    16     451.00        279.62             4.51

Ingreso medio por revisión: 333 €
Beneficio neto medio:       206 €


## 5. Estrategias Comerciales

| Segmento | Acción | Contacto | Adicional | Bono | Response Rate |
|----------|--------|----------|-----------|------|---------------|
| MUY_ALTO | Email + Lavado completo | 0,20 € | — | 5 € | 10 % |
| ALTO | SMS + Regalo | 0,50 € | 6 € | 30 € | 55 % |
| MEDIO | Llamada + Recogida + Lavado + Neumáticos | 0,40 € | 100 € | 50 € | 60 % |
| BAJO | SMS + Pack 5 rev. (−15 %) | 0,50 € | — | 10 € | 45 % |
| MUY_BAJO | Email + Pack 5 rev. (−15 %) | 0,20 € | — | — | 40 % |

> Los segmentos BAJO y MUY_BAJO reciben la oferta del **Pack 5 revisiones con un 15 % de descuento** para bloquear su fidelidad durante los próximos 3–5 años.

In [40]:
# ── Configuración de estrategias ────────────────────────────
COSTE_CONTACTO  = {'MUY_ALTO': 0.20, 'ALTO': 0.50, 'MEDIO': 0.40, 'BAJO': 0.50, 'MUY_BAJO': 0.20}
COSTE_ADICIONAL = {'MUY_ALTO': 0.00, 'ALTO': 6.00, 'MEDIO': 100.00, 'BAJO': 0.00, 'MUY_BAJO': 0.00}
RESPONSE_RATE   = {'MUY_ALTO': 0.15, 'ALTO': 0.60, 'MEDIO': 0.65, 'BAJO': 0.60, 'MUY_BAJO': 0.60}
BONOS_FIJOS     = {'MUY_ALTO': 5.00, 'ALTO': 27.00, 'MEDIO': 50.00, 'BAJO': 10.00, 'MUY_BAJO': 0.00}

clientes['Bono_€']         = clientes['segmento'].map(BONOS_FIJOS)
clientes['Descuento_€']    = clientes['Bono_€']
clientes['RR']             = clientes['segmento'].map(RESPONSE_RATE)
clientes['Coste_Contacto'] = clientes['segmento'].map(COSTE_CONTACTO)
clientes['Coste_Adicional']= clientes['segmento'].map(COSTE_ADICIONAL)
clientes['Beneficio_Neto'] = clientes['Margen_Bruto'] - clientes['Bono_€']

resumen_est = pd.DataFrame({
    'Segmento'   : list(BONOS_FIJOS.keys()),
    'N_clientes' : [(clientes['segmento'] == s).sum() for s in BONOS_FIJOS],
    'Contacto_€' : list(COSTE_CONTACTO.values()),
    'Adicional_€': list(COSTE_ADICIONAL.values()),
    'Bono_€'     : list(BONOS_FIJOS.values()),
    'RR_%'       : [int(v*100) for v in RESPONSE_RATE.values()],
})
print(resumen_est.to_string(index=False))

Segmento  N_clientes  Contacto_€  Adicional_€  Bono_€  RR_%
MUY_ALTO         260        0.20         0.00    5.00    15
    ALTO        4343        0.50         6.00   27.00    60
   MEDIO        3916        0.40       100.00   50.00    65
    BAJO          31        0.50         0.00   10.00    60
MUY_BAJO        1450        0.20         0.00    0.00    60


## 6. ROI — Estrategia Solo-Churn

Calculamos el retorno sobre la inversión para cada segmento aplicando la estrategia definida arriba.

- Se contacta a **todos** los clientes del segmento (coste de contacto)
- Solo los que **responden** (según el Response Rate) generan coste adicional, marketing y bono
- El margen bruto se calcula únicamente sobre los que responden

$$\text{ROI} = \frac{\text{Margen\_bruto} - \text{Costes\_campaña}}{\text{Costes\_campaña}}$$

In [41]:
# ── ROI por segmento ────────────────────────────────────────
resultados_roi = []

for segmento in ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']:
    seg = clientes[clientes['segmento'] == segmento]
    if len(seg) == 0:
        continue

    n_total     = len(seg)
    rr          = RESPONSE_RATE[segmento]
    n_responden = n_total * rr

    coste_contacto_total  = n_total     * COSTE_CONTACTO[segmento]
    coste_adicional_total = n_responden * COSTE_ADICIONAL[segmento]
    coste_marketing_total = n_responden * seg['Coste_Marketing'].mean()
    coste_descuentos      = n_responden * seg['Descuento_€'].mean()

    costes_campana     = coste_contacto_total + coste_adicional_total + coste_marketing_total + coste_descuentos
    margen_bruto_total = n_responden * seg['Margen_Bruto'].mean()
    beneficio_neto     = margen_bruto_total - costes_campana
    roi                = beneficio_neto / costes_campana if costes_campana > 0 else 0

    resultados_roi.append({
        'Segmento'      : segmento,
        'N_Clientes'    : n_total,
        'RR_%'          : int(rr * 100),
        'N_Responden'   : int(n_responden),
        'Costes_Total_€': round(costes_campana, 0),
        'Margen_Bruto_€': round(margen_bruto_total, 0),
        'Beneficio_€'   : round(beneficio_neto, 0),
        'ROI'           : round(roi, 4),
    })

df_roi = pd.DataFrame(resultados_roi)
costes_total    = df_roi['Costes_Total_€'].sum()
beneficio_total = df_roi['Beneficio_€'].sum()
roi_global      = beneficio_total / costes_total if costes_total > 0 else 0

print(df_roi.to_string(index=False))
print(f'\nInversión total:  {costes_total:>10,.0f} €')
print(f'Beneficio neto:   {beneficio_total:>10,.0f} €')
print(f'ROI global:       {roi_global:>10.4f}x')

Segmento  N_Clientes  RR_%  N_Responden  Costes_Total_€  Margen_Bruto_€  Beneficio_€   ROI
MUY_ALTO         260    15           39          387.00        8,687.00     8,300.00 21.44
    ALTO        4343    60         2605       96,350.00      507,570.00   411,221.00  4.27
   MEDIO        3916    65         2545      392,289.00      552,584.00   160,295.00  0.41
    BAJO          31    60           18          275.00        4,566.00     4,291.00 15.60
MUY_BAJO        1450    60          870        3,220.00      181,675.00   178,455.00 55.42

Inversión total:     492,521 €
Beneficio neto:      762,562 €
ROI global:           1.5483x


## 7. Visualizaciones — Estrategia Solo-Churn

Representamos el impacto económico de la campaña: ROI por segmento, inversión vs beneficio, distribución de clientes y desglose de costes.

In [42]:
# ── Dashboard: Estrategia Solo-Churn ────────────────────────
# 4 gráficos en un solo layout: ROI, Costes, Distribución/RR y Waterfall

from plotly.subplots import make_subplots

fig_dash = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'ROI por segmento',
        'Desglose de costes por segmento',
        'Distribución de clientes',
        'Tasa de captación (Response Rate)',
    ),
    specs=[
        [{'type': 'bar'},  {'type': 'bar'}],
        [{'type': 'pie'},  {'type': 'bar'}],
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.10,
)

# ── [1,1] ROI por segmento ───────────────────────────────────
colores_roi = ['#27ae60' if r >= 3 else '#f39c12' if r >= 1 else '#e74c3c' for r in df_roi['ROI']]
fig_dash.add_trace(go.Bar(
    x=df_roi['Segmento'], y=df_roi['ROI'],
    marker_color=colores_roi,
    text=[f"{v:.2f}x" for v in df_roi['ROI']],
    textposition='outside',
    showlegend=False,
), row=1, col=1)
fig_dash.add_hline(y=1, line_dash='dash', line_color='white', line_width=1,
                   annotation_text='Break-even', annotation_position='top right',
                   row=1, col=1)

# ── [1,2] Desglose de costes (barras apiladas) ───────────────
fig_dash.add_trace(go.Bar(
    name='Contacto', x=df_roi['Segmento'], y=df_roi['Costes_Total_€'] * 0,  # placeholder
    marker_color='#95a5a6', showlegend=True,
), row=1, col=2)

# Recalcular columnas de coste (pueden tener nombres distintos según versión)
coste_cols = [c for c in df_roi.columns if 'Coste' in c and c != 'Costes_Total_€']

contacto_vals  = df_roi['Costes_Total_€'] * 0  # fallback zeros
adicional_vals = df_roi['Costes_Total_€'] * 0
marketing_vals = df_roi['Costes_Total_€'] * 0
bonos_vals     = df_roi['Costes_Total_€'] * 0

# Reconstruir desde los diccionarios originales
seg_list = df_roi['Segmento'].tolist()
contacto_vals  = [len(clientes[clientes['segmento']==s]) * COSTE_CONTACTO[s] for s in seg_list]
adicional_vals = [len(clientes[clientes['segmento']==s]) * RESPONSE_RATE[s] * COSTE_ADICIONAL[s] for s in seg_list]
marketing_vals = [len(clientes[clientes['segmento']==s]) * RESPONSE_RATE[s] * clientes[clientes['segmento']==s]['Coste_Marketing'].mean() for s in seg_list]
bonos_vals     = [len(clientes[clientes['segmento']==s]) * RESPONSE_RATE[s] * BONOS_FIJOS[s] for s in seg_list]

# Eliminar el trace placeholder y añadir los reales
fig_dash.data = fig_dash.data[:-1]  # quitar placeholder

for vals, nombre, color in [
    (contacto_vals,  'Contacto',   '#95a5a6'),
    (adicional_vals, 'Servicios',  '#3498db'),
    (marketing_vals, 'Marketing',  '#9b59b6'),
    (bonos_vals,     'Bonos',      '#e67e22'),
]:
    fig_dash.add_trace(go.Bar(
        name=nombre, x=seg_list, y=vals,
        marker_color=color,
        text=[f"{v:,.0f}€" for v in vals],
        textposition='inside',
    ), row=1, col=2)

fig_dash.update_layout(barmode='stack')

# ── [2,1] Distribución de clientes (pie) ────────────────────
fig_dash.add_trace(go.Pie(
    labels=df_roi['Segmento'],
    values=df_roi['N_Clientes'],
    marker=dict(colors=['#e74c3c', '#e67e22', '#f39c12', '#27ae60', '#3498db']),
    textinfo='label+percent',
    textfont=dict(size=11),
    showlegend=False,
), row=2, col=1)

# ── [2,2] Response Rate por segmento ────────────────────────
fig_dash.add_trace(go.Bar(
    x=df_roi['Segmento'], y=df_roi['RR_%'],
    marker_color='#3498db',
    text=[f"{v}%" for v in df_roi['RR_%']],
    textposition='outside',
    showlegend=False,
), row=2, col=2)

# ── Layout general ───────────────────────────────────────────
fig_dash.update_layout(
    title=dict(
        text='<b>Dashboard — Estrategia Solo-Churn</b>',
        font=dict(size=20, color='white'),
    ),
    template='plotly_dark',
    height=900,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig_dash.update_yaxes(title_text='ROI', row=1, col=1)
fig_dash.update_yaxes(title_text='€', row=1, col=2)
fig_dash.update_yaxes(title_text='%', row=2, col=2)

fig_dash.show()


In [43]:
import plotly.graph_objects as go

# ══════════════════════════════════════════════════════════════════════════════
# 1️⃣ GAUGE CHART: El indicador clave de éxito (ROI)
# ══════════════════════════════════════════════════════════════════════════════

fig_gauge = go.Figure(go.Indicator(
    mode = "gauge+number",
    value = roi_global,
    domain = {'x': [0, 1], 'y': [0, 1]},
    title = {
        'text': "<b>ROI GLOBAL DE LA ESTRATEGIA</b><br><span style='font-size:0.8em;color:gray'>Retorno por cada 1€ invertido</span>",
        'font': {'size': 24, 'family': 'Arial Black'}
    },
    number = {
        'font': {'size': 70, 'color': "#27ae60", 'family': 'Arial Black'}, 
        'suffix': "x", # Para que salga como 5.00x
        'valueformat': '.2f'
    },
    gauge = {
        'axis': {'range': [0, 6], 'tickwidth': 1, 'tickcolor': "#2c3e50"},
        'bar': {'color': "#27ae60", 'thickness': 0.8}, # El color de la aguja/barra
        'bgcolor': "white",
        'borderwidth': 1,
        'bordercolor': "#bdc3c7",
        'steps': [
            {'range': [0, 1], 'color': '#ffcccc'}, # Zona Roja: Pérdidas
            {'range': [1, 2], 'color': '#ffe5d9'}, # Zona Naranja: Recuperación
            {'range': [2, 4], 'color': '#fff3cd'}, # Zona Amarilla: Rentable
            {'range': [4, 6], 'color': '#d4edda'}  # Zona Verde: Excelente
        ],
        'threshold': {
            'line': {'color': "black", 'width': 5},
            'thickness': 0.75,
            'value': 3.0 # La marca del objetivo mínimo
        }
    }
))

# Ajustes de diseño para que se vea impecable
fig_gauge.update_layout(
    height=500,
    template='plotly_white',
    margin={'t': 100, 'b': 50, 'l': 50, 'r': 50},
    font=dict(family="Arial", color="#34495e")
)

# Añadimos un cuadro con el resumen de dinero debajo (opcional, pero muy pro)
fig_gauge.add_annotation(
    text=f"<b>Inversión Total:</b> {costes_total:,.0f}€  |  <b>Beneficio Neto:</b> {beneficio_total:,.0f}€",
    xref="paper", yref="paper",
    x=0.5, y=-0.05, showarrow=False,
    font=dict(size=16, color="#2c3e50"),
    bordercolor="#27ae60", borderpad=8, bgcolor="#f8f9fa", borderwidth=2
)

fig_gauge.show()

---

## 8. Estrategia Avanzada: Churn × CLTV

La estrategia anterior asigna la acción **solo en función del riesgo de churn**. Esto tiene un problema: no distingue entre un cliente que vale 200 € y uno que vale 3.000 €, aunque ambos tengan la misma probabilidad de abandonar.

La solución es añadir una segunda dimensión: el **Customer Lifetime Value (CLTV)**, que estima el beneficio total esperado del cliente durante las próximas revisiones, ponderado por su probabilidad de seguir activo.

Cruzando las dos dimensiones obtenemos una **matriz 5 × 3** (5 niveles de riesgo × 3 niveles de valor) con 15 combinaciones, cada una con una acción y una inversión diferente. El principio es simple: **invertir más donde más se puede ganar**, no donde más se puede perder.

In [44]:
# ══════════════════════════════════════════════════════════════════════════════
# PARTE 3: Cálculo del CLTV (Fórmula del Profesor)
# ══════════════════════════════════════════════════════════════════════════════
#
# CLTV = Σ(k=1..N) [ C(n+k) × 0.62 × (1 - p_churn)^k ]
#
# Donde:
#   C(n+k) = BASE × (1+α)^(n+k)   → ingreso de la revisión futura k
#   0.62                           → beneficio neto (100% - 38% costes)
#   (1 - p_churn)^k                → probabilidad de que el cliente siga activo
#   n = Revisiones actuales, N = 20 revisiones futuras máx.
#
# Forma cerrada: BASE × 0.62 × (1+α)^n × q×(1-q^N)/(1-q)
# donde q = (1+α)×(1-p_churn)
# ──────────────────────────────────────────────────────────────────────────────

N_FUTURAS = 20

clientes_cltv = clientes.copy()

def calcular_cltv(row):
    base  = row['Mantenimiento_medio']
    alpha = 0.07 if row['Modelo'] in ['A', 'B'] else 0.10
    n     = row['Revisiones']
    p     = min(row['prob_churn'], 0.999)   # evitar p=1
    q     = (1 + alpha) * (1 - p)
    if abs(q - 1) < 1e-9:
        total = N_FUTURAS
    else:
        total = q * (1 - q**N_FUTURAS) / (1 - q)
    return base * ((1 + alpha)**n) * 0.62 * total

clientes_cltv['CLTV'] = clientes_cltv.apply(calcular_cltv, axis=1)

# Clasificación en 3 niveles por percentil
p33 = clientes_cltv['CLTV'].quantile(0.33)
p66 = clientes_cltv['CLTV'].quantile(0.66)

def nivel_cltv(cltv):
    if cltv > p66: return 'ALTO'
    if cltv > p33: return 'MEDIO'
    return 'BAJO'

clientes_cltv['cltv_nivel'] = clientes_cltv['CLTV'].apply(nivel_cltv)

print(f"Percentiles CLTV: P33={p33:,.0f}€  |  P66={p66:,.0f}€")
print(f"\nDistribución de niveles CLTV:")
for nivel in ['BAJO', 'MEDIO', 'ALTO']:
    n = (clientes_cltv['cltv_nivel'] == nivel).sum()
    cltv_med = clientes_cltv[clientes_cltv['cltv_nivel'] == nivel]['CLTV'].median()
    print(f"  {nivel:5s}: {n:5,} clientes  |  CLTV mediano: {cltv_med:,.0f}€")

print(f"\nEjemplos representativos por segmento de churn:")
for seg in ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']:
    ej = clientes_cltv[clientes_cltv['segmento'] == seg][['Modelo','prob_churn','C_n','CLTV','cltv_nivel']].head(2)
    print(f"\n  {seg}:")
    print(ej.to_string(index=False))

Percentiles CLTV: P33=113€  |  P66=197€

Distribución de niveles CLTV:
  BAJO : 3,300 clientes  |  CLTV mediano: 90€
  MEDIO: 3,314 clientes  |  CLTV mediano: 149€
  ALTO : 3,386 clientes  |  CLTV mediano: 330€

Ejemplos representativos por segmento de churn:

  MUY_ALTO:
Modelo  prob_churn    C_n  CLTV cltv_nivel
     I        0.82 408.10 57.03       BAJO
     A        0.82 267.50 36.74       BAJO

  ALTO:
Modelo  prob_churn    C_n   CLTV cltv_nivel
     A        0.64 267.50  95.24       BAJO
     H        0.65 388.30 135.59      MEDIO

  MEDIO:
Modelo  prob_churn    C_n   CLTV cltv_nivel
     B        0.59 281.41 129.65      MEDIO
     B        0.57 281.41 138.90      MEDIO

  BAJO:
Modelo  prob_churn    C_n     CLTV cltv_nivel
     H        0.22 388.30 1,218.88       ALTO
     I        0.27 408.10   939.77       ALTO

  MUY_BAJO:
Modelo  prob_churn    C_n     CLTV cltv_nivel
     D        0.05 319.00 5,949.92       ALTO
     I        0.09 408.10 4,601.26       ALTO


In [45]:
# ── Asignación de estrategia 2D: Churn × CLTV ──────────────
PACK_5_REV = ' + Pack 5 rev. (-10%) + 1.000€ prox. vehiculo'

ESTRATEGIA_2D = {
    # MUY_ALTO: churn >80%, inversión mínima
    ('MUY_ALTO', 'BAJO'):  (0.05,  0.00,  5.00, 0.13, 'Email + Bono 5€'),
    ('MUY_ALTO', 'MEDIO'): (0.05,  0.00,  5.00, 0.13, 'Email + Bono 5€'),
    ('MUY_ALTO', 'ALTO'):  (0.05, 0.00,  5.00, 0.13, 'Email + Bono 5€'),

    # ALTO: churn 60-80%, acción moderada sin lavado, bonos -10%
    ('ALTO', 'BAJO'):  (0.10,  0.00,  5.00, 0.30, 'SMS + Bono 5€'),
    ('ALTO', 'MEDIO'): (0.10,  6.00, 25.00, 0.60, 'SMS + Regalo + Bono 25€'),
    ('ALTO', 'ALTO'):  (0.10,  6.00, 35.00, 0.65, 'SMS + Regalo + Bono 35€'),

    # MEDIO: churn 40-60%, estrategia premium vía llamada
    ('MEDIO', 'BAJO'):  (0.40,  0.00, 10.00, 0.35, 'Llamada + Bono 10€'),
    ('MEDIO', 'MEDIO'): (0.40, 15.00, 45.00, 0.65, 'Llamada + Lavado + Bono 45€'),
    ('MEDIO', 'ALTO'):  (0.40, 65.00, 55.00, 0.70, 'Llamada + Pack (Lav+Neum) + Bono 55€'),

    # BAJO: churn 20-40%, fidelización
    ('BAJO', 'BAJO'):  (0.06,  0.00,  0.00, 0.15, 'Email recordatorio revision' + PACK_5_REV),
    ('BAJO', 'MEDIO'): (0.10,  0.00, 10.00, 0.50, 'SMS + Bono 10€' + PACK_5_REV),
    ('BAJO', 'ALTO'):  (0.10,  0.00, 10.00, 0.55, 'SMS + Bono 10€' + PACK_5_REV),

    # MUY_BAJO: churn <20%, retención garantizada
    ('MUY_BAJO', 'BAJO'):  (0.05,  0.00,  0.00, 0.10, 'Email recordatorio revision' + PACK_5_REV),
    ('MUY_BAJO', 'MEDIO'): (0.05,  0.00,  0.00, 0.45, 'Email' + PACK_5_REV),
    ('MUY_BAJO', 'ALTO'):  (0.05,  0.00,  0.00, 0.50, 'Email' + PACK_5_REV),
}

def asignar_estrategia_cltv(row):
    return ESTRATEGIA_2D.get((row['segmento'], row['cltv_nivel']), (0.20, 0.00, 10.00, 0.20, 'Estandar'))

estrategias = clientes_cltv.apply(asignar_estrategia_cltv, axis=1, result_type='expand')
estrategias.columns = ['CC_cltv', 'CA_cltv', 'Bono_cltv', 'RR_cltv', 'Accion_cltv']
clientes_cltv = pd.concat([
    clientes_cltv.drop(columns=['CC_cltv','CA_cltv','Bono_cltv','RR_cltv','Accion_cltv'], errors='ignore'),
    estrategias
], axis=1)

filas = []
for seg in ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']:
    for nivel in ['BAJO', 'MEDIO', 'ALTO']:
        cc, ca, bono, rr, accion = ESTRATEGIA_2D[(seg, nivel)]
        n = ((clientes_cltv['segmento'] == seg) & (clientes_cltv['cltv_nivel'] == nivel)).sum()
        filas.append({'Churn': seg, 'CLTV': nivel, 'N': n,
                      'Contacto€': cc, 'Adicional€': ca, 'Bono€': bono,
                      'RR%': int(rr*100), 'Accion': accion})

df_matriz = pd.DataFrame(filas)
display(df_matriz[['Churn','CLTV','N','Contacto€','Adicional€','Bono€','RR%','Accion']])

print('\nPack 5 revisiones (BAJO y MUY_BAJO):')
print('  - 5 revisiones con ~10% de descuento sobre tarifa individual')
print('  - Bono de 1.000€ aplicable a la compra del próximo vehículo')
print('  - Objetivo: asegurar fidelidad durante los próximos 3-5 años')

,Churn,CLTV,N,Contacto€,Adicional€,Bono€,RR%,Accion
0,MUY_ALTO,BAJO,260,0.05,0.00,5.00,13,Email + Bono 5€
1,MUY_ALTO,MEDIO,0,0.05,0.00,5.00,13,Email + Bono 5€
2,MUY_ALTO,ALTO,0,0.05,0.00,5.00,13,Email + Bono 5€
3,ALTO,BAJO,3040,0.10,0.00,5.00,30,SMS + Bono 5€
4,ALTO,MEDIO,1303,0.10,6.00,25.00,60,SMS + Regalo + Bono 25€
5,ALTO,ALTO,0,0.10,6.00,35.00,65,SMS + Regalo + Bono 35€
6,MEDIO,BAJO,0,0.40,0.00,10.00,35,Llamada + Bono 10€
7,MEDIO,MEDIO,2011,0.40,15.00,45.00,65,Llamada + Lavado + Bono 45€
8,MEDIO,ALTO,1905,0.40,65.00,55.00,70,Llamada + Pack (Lav+Neum) + Bono 55€
9,BAJO,BAJO,0,0.06,0.00,0.00,15,Email recordatorio revision + Pack 5 rev. (-10...



Pack 5 revisiones (BAJO y MUY_BAJO):
  - 5 revisiones con ~10% de descuento sobre tarifa individual
  - Bono de 1.000€ aplicable a la compra del próximo vehículo
  - Objetivo: asegurar fidelidad durante los próximos 3-5 años


In [46]:
# ── ROI con estrategia CLTV (por celda de la matriz 2D) ────
# Sin filtro de CLTV_MIN por ahora — N=10 ya recalibra la distribución

resultados_cltv = []

for seg in ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']:
    for nivel in ['BAJO', 'MEDIO', 'ALTO']:
        grupo = clientes_cltv[
            (clientes_cltv['segmento'] == seg) &
            (clientes_cltv['cltv_nivel'] == nivel)
        ]
        if len(grupo) == 0:
            continue

        cc, ca, bono, rr, accion = ESTRATEGIA_2D[(seg, nivel)]
        n_total     = len(grupo)
        n_responden = n_total * rr

        costes_campana     = (n_total * cc
                              + n_responden * ca
                              + n_responden * grupo['Coste_Marketing'].mean()
                              + n_responden * bono)
        margen_bruto_total = n_responden * grupo['Margen_Bruto'].mean()
        beneficio_neto     = margen_bruto_total - costes_campana
        roi                = beneficio_neto / costes_campana if costes_campana > 0 else 0

        resultados_cltv.append({
            'Churn': seg, 'CLTV': nivel, 'N': n_total,
            'RR%': int(rr * 100), 'N_Resp': int(n_responden),
            'Costes €': round(costes_campana, 0),
            'Margen Bruto €': round(margen_bruto_total, 0),
            'Beneficio €': round(beneficio_neto, 0),
            'ROI': round(roi, 4),
        })

df_roi_cltv = pd.DataFrame(resultados_cltv)
costes_total_cltv    = df_roi_cltv['Costes €'].sum()
beneficio_total_cltv = df_roi_cltv['Beneficio €'].sum()
roi_global_cltv      = beneficio_total_cltv / costes_total_cltv if costes_total_cltv > 0 else 0

display(df_roi_cltv)
print(f'Inversión total:  {costes_total_cltv:>10,.0f} €')
print(f'Beneficio neto:   {beneficio_total_cltv:>10,.0f} €')
print(f'ROI global:       {roi_global_cltv:>10.4f}x')
print(f'Clientes contactados:   {df_roi_cltv["N"].sum():,}')
print(f'Clientes que responden: {int(df_roi_cltv["N_Resp"].sum()):,}')

,Churn,CLTV,N,RR%,N_Resp,Costes €,Margen Bruto €,Beneficio €,ROI
0,MUY_ALTO,BAJO,260,13,33,303.00,"7,529.00","7,225.00",23.81
1,ALTO,BAJO,3040,30,912,"7,604.00","169,857.00","162,253.00",21.34
2,ALTO,MEDIO,1303,60,781,"27,073.00","167,857.00","140,784.00",5.20
3,MEDIO,MEDIO,2011,65,1307,"83,265.00","249,964.00","166,699.00",2.00
4,MEDIO,ALTO,1905,70,1333,"166,038.00","325,899.00","159,860.00",0.96
5,BAJO,ALTO,31,55,17,241.00,"4,186.00","3,945.00",16.36
6,MUY_BAJO,ALTO,1450,50,725,"2,514.00","151,396.00","148,881.00",59.21


Inversión total:     287,038 €
Beneficio neto:      789,647 €
ROI global:           2.7510x
Clientes contactados:   10,000
Clientes que responden: 5,108


In [47]:
# Proyeccion de revisiones futuras por modelo
# C(n) = BASE x (1+alfa)^n  |  Beneficio = C(n) x 0.62

N_MAX = 7
DESCUENTO_PACK = 0.10
BONO_VEHICULO  = 1000.0

modelos_info = (clientes_cltv
    .groupby('Modelo')['Mantenimiento_medio']
    .first().sort_index().to_dict()
)

# Churn medio real por modelo
churn_por_modelo = clientes_cltv.groupby('Modelo')['prob_churn'].mean().round(3)

# Tabla pivot: filas=Modelo, columnas=Rev 1..7 (beneficio no acumulado)
pivot_rows = []
pack_rows  = []

for modelo, base in modelos_info.items():
    alpha  = 0.07 if modelo in ['A', 'B'] else 0.10
    n_cli  = (clientes_cltv['Modelo'] == modelo).sum()
    p_churn = churn_por_modelo[modelo]

    row = {'Modelo': modelo, 'N clientes': n_cli, 'Churn medio': p_churn}
    for n in range(1, N_MAX + 1):
        benef = base * ((1 + alpha) ** n) * 0.62
        row['Rev ' + str(n)] = round(benef, 1)
    pivot_rows.append(row)

    ingreso_5   = sum(base * ((1 + alpha) ** n) for n in range(1, 6))
    precio_pack = ingreso_5 * (1 - DESCUENTO_PACK)
    benef_pack  = precio_pack * 0.62
    pack_rows.append({
        'Modelo': modelo,
        'Ingreso 5 rev. €': round(ingreso_5, 2),
        'Precio pack (-10%) €': round(precio_pack, 2),
        'Benef. taller pack €': round(benef_pack, 2),
        'Tras bono vehiculo €': round(benef_pack - BONO_VEHICULO, 2),
    })

df_pivot = pd.DataFrame(pivot_rows)
df_pack  = pd.DataFrame(pack_rows)

# Ordenar por churn medio (los que mas se quedan primero)
# ordenado A-K (orden natural del dict)

print("Beneficio neto por revisión y modelo (€):")
display(df_pivot)

print("\nPack 5 revisiones por modelo:")
display(df_pack)


Beneficio neto por revisión y modelo (€):


,Modelo,N clientes,Churn medio,Rev 1,Rev 2,Rev 3,Rev 4,Rev 5,Rev 6,Rev 7
0,A,1277,0.61,165.80,177.50,189.90,203.20,217.40,232.60,248.90
1,B,2779,0.54,174.50,186.70,199.80,213.70,228.70,244.70,261.80
2,C,705,0.50,188.20,207.10,227.80,250.50,275.60,303.10,333.50
3,D,1547,0.49,197.80,217.60,239.30,263.20,289.60,318.50,350.40
4,E,118,0.55,208.00,228.80,251.70,276.90,304.50,335.00,368.50
5,F,2,0.59,218.20,240.10,264.10,290.50,319.50,351.50,386.60
6,G,7,0.63,229.20,252.10,277.30,305.00,335.50,369.10,406.00
7,H,473,0.69,240.70,264.80,291.30,320.40,352.50,387.70,426.50
8,I,2625,0.53,253.00,278.30,306.20,336.80,370.40,407.50,448.20
9,J,451,0.33,266.00,292.60,321.80,354.00,389.40,428.40,471.20



Pack 5 revisiones por modelo:


,Modelo,Ingreso 5 rev. €,Precio pack (-10%) €,Benef. taller pack €,Tras bono vehiculo €
0,A,"1,538.32","1,384.49",858.38,-141.62
1,B,"1,618.32","1,456.48",903.02,-96.98
2,C,"1,853.51","1,668.16","1,034.26",34.26
3,D,"1,947.53","1,752.77","1,086.72",86.72
4,E,"2,048.26","1,843.43","1,142.93",142.93
5,F,"2,149.00","1,934.10","1,199.14",199.14
6,G,"2,256.44","2,030.80","1,259.10",259.10
7,H,"2,370.61","2,133.55","1,322.80",322.80
8,I,"2,491.49","2,242.34","1,390.25",390.25
9,J,"2,619.09","2,357.18","1,461.45",461.45


In [48]:
# Curva de beneficio neto por revision y modelo

# Construir df largo: una fila por (modelo, revision)
filas_rev = []
for modelo, base in modelos_info.items():
    alpha = 0.07 if modelo in ['A', 'B'] else 0.10
    for n in range(1, N_MAX + 1):
        filas_rev.append({
            'Modelo':        modelo,
            'Revision':      n,
            'Beneficio neto': round(base * ((1 + alpha) ** n) * 0.62, 2),
        })

df_rev = pd.DataFrame(filas_rev)

# Paleta de colores para cada modelo
colores = [
    '#2ecc71','#3498db','#9b59b6','#e67e22','#e74c3c',
    '#1abc9c','#f39c12','#d35400','#27ae60','#2980b9','#8e44ad'
]

fig = go.Figure()

for i, modelo in enumerate(sorted(modelos_info.keys())):
    subset = df_rev[df_rev['Modelo'] == modelo]
    churn_m = round(churn_por_modelo[modelo] * 100, 1)
    fig.add_trace(go.Scatter(
        x=subset['Revision'],
        y=subset['Beneficio neto'],
        mode='lines+markers',
        name='Modelo ' + modelo + ' (churn ' + str(churn_m) + '%)',
        line=dict(color=colores[i % len(colores)], width=2),
        marker=dict(size=7),
    ))

# Linea vertical en revision 5: umbral del pack
fig.add_vline(
    x=5, line_dash='dash', line_color='#e74c3c', line_width=2,
    annotation_text='Rev. 5: umbral pack + dto. 2º vehiculo',
    annotation_position='top left',
    annotation_font_color='#e74c3c',
)

fig.update_layout(
    title=dict(
        text='<b>Evolucion del beneficio neto por revision y modelo</b>',
        font=dict(size=18, color='#2c3e50')
    ),
    xaxis=dict(title='Numero de revision', tickmode='linear', dtick=1),
    yaxis=dict(title='Beneficio neto (€)'),
    template='plotly_white',
    height=500,
    legend=dict(orientation='v', x=1.01, y=1),
)

fig.show()

In [49]:
# ── Comparativa: Solo-Churn vs Churn × CLTV ─────────────────
from plotly.subplots import make_subplots

# Tabla comparativa
df_comp_tabla = pd.DataFrame({
    'Métrica': ['Inversión total (€)', 'Beneficio neto (€)', 'ROI global', 'Clientes que responden'],
    'Solo-Churn': [
        f"{costes_total:,.0f}",
        f"{beneficio_total:,.0f}",
        f"{roi_global:.4f}x",
        f"{int(df_roi['N_Responden'].sum()):,}",
    ],
    'Churn × CLTV': [
        f"{costes_total_cltv:,.0f}",
        f"{beneficio_total_cltv:,.0f}",
        f"{roi_global_cltv:.4f}x",
        f"{int(df_roi_cltv['N_Resp'].sum()):,}",
    ],
    'Diferencia': [
        f"{costes_total_cltv - costes_total:+,.0f}",
        f"{beneficio_total_cltv - beneficio_total:+,.0f}",
        f"{roi_global_cltv - roi_global:+.4f}",
        f"{int(df_roi_cltv['N_Resp'].sum()) - int(df_roi['N_Responden'].sum()):+,}",
    ],
})
display(df_comp_tabla)

# ── Dashboard: ROI comparativo + Heatmap ────────────────────
segmentos_orden = ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']

roi_churn_seg = df_roi.set_index('Segmento')['ROI']
roi_cltv_seg  = (df_roi_cltv.groupby('Churn')['Beneficio €'].sum()
                 / df_roi_cltv.groupby('Churn')['Costes €'].sum())

roi_churn_vals = [roi_churn_seg.get(s, 0) for s in segmentos_orden]
roi_cltv_vals  = [roi_cltv_seg.get(s, 0) for s in segmentos_orden]

pivot_roi = df_roi_cltv.pivot(index='Churn', columns='CLTV', values='ROI').reindex(
    index=segmentos_orden, columns=['BAJO', 'MEDIO', 'ALTO']
).fillna(0)

fig_final = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'ROI por segmento: Solo-Churn vs Churn × CLTV',
        'Heatmap de ROI — Matriz Churn × CLTV',
    ),
    column_widths=[0.5, 0.5],
    horizontal_spacing=0.12,
)

# Barras comparativas
fig_final.add_trace(go.Bar(
    name='Solo-Churn', x=segmentos_orden, y=roi_churn_vals,
    marker_color='#3498db',
    text=[f"{v:.2f}x" for v in roi_churn_vals], textposition='outside',
), row=1, col=1)
fig_final.add_trace(go.Bar(
    name='Churn × CLTV', x=segmentos_orden, y=roi_cltv_vals,
    marker_color='#27ae60',
    text=[f"{v:.2f}x" for v in roi_cltv_vals], textposition='outside',
), row=1, col=1)
fig_final.add_hline(y=1, line_dash='dash', line_color='red', line_width=1,
                    annotation_text='Break-even', row=1, col=1)

# Heatmap
fig_final.add_trace(go.Heatmap(
    z=pivot_roi.values,
    x=['CLTV Bajo', 'CLTV Medio', 'CLTV Alto'],
    y=segmentos_orden,
    colorscale='RdYlGn',
    text=[[f"{v:.1f}x" for v in row] for row in pivot_roi.values],
    texttemplate="%{text}",
    textfont=dict(size=13, color='black'),
    colorbar=dict(title='ROI', x=1.01),
    showscale=True,
), row=1, col=2)

fig_final.update_layout(
    title=dict(
        text='<b>Comparativa de estrategias</b>',
        font=dict(size=18, color='#2c3e50'),
    ),
    barmode='group',
    template='plotly_white',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=0.5),
)
fig_final.update_yaxes(title_text='ROI', row=1, col=1)
fig_final.update_yaxes(title_text='Segmento Churn', row=1, col=2)
fig_final.show()


,Métrica,Solo-Churn,Churn × CLTV,Diferencia
0,Inversión total (€),"492,521","287,038","-205,483"
1,Beneficio neto (€),"762,562","789,647","+27,085"
2,ROI global,1.5483x,2.7510x,+1.2027
3,Clientes que responden,"6,077","5,108",-969


In [50]:
# ── Gauges comparativos: ROI Solo-Churn vs ROI Churn×CLTV ──
fig_gauges = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}]],
    subplot_titles=('ROI — Estrategia Solo-Churn', 'ROI — Estrategia Churn × CLTV'),
)

def gauge_trace(value, max_val=6):
    return go.Indicator(
        mode='gauge+number',
        value=value,
        number={'font': {'size': 52, 'color': '#27ae60'}, 'suffix': 'x', 'valueformat': '.2f'},
        gauge={
            'axis': {'range': [0, max_val]},
            'bar': {'color': '#27ae60', 'thickness': 0.8},
            'steps': [
                {'range': [0, 1],        'color': '#ffcccc'},
                {'range': [1, 2],        'color': '#ffe5d9'},
                {'range': [2, 4],        'color': '#fff3cd'},
                {'range': [4, max_val],  'color': '#d4edda'},
            ],
            'threshold': {'line': {'color': 'black', 'width': 4}, 'thickness': 0.75, 'value': 3.0},
        },
    )

fig_gauges.add_trace(gauge_trace(roi_global),      row=1, col=1)
fig_gauges.add_trace(gauge_trace(roi_global_cltv), row=1, col=2)

fig_gauges.update_layout(
    title=dict(text='<b>Comparativa de ROI global entre estrategias</b>', font=dict(size=18, color='#2c3e50')),
    template='plotly_white',
    height=380,
    margin={'t': 80, 'b': 40, 'l': 40, 'r': 40},
)
fig_gauges.show()

---

## 9. Análisis de Rentabilidad: Descuento 2º Vehículo

El bono de **1.000 €** para el 2º vehículo se ofrece a los segmentos **BAJO** y **MUY_BAJO** como parte del Pack 5 revisiones.  
Su coste real depende de cuántos clientes lo **redimen efectivamente** al comprar un nuevo vehículo.

Modelamos tres escenarios de tasa de redención:

| Escenario  | % de compradores del pack que redimen el bono |
|------------|----------------------------------------------|
| Pesimista  | 10 %                                         |
| Base       | 20 %                                         |
| Optimista  | 30 %                                         |

> **Nota:** el bono solo se paga cuando el cliente compra el 2º vehículo, no en el momento de firmar el pack.  
> Por tanto, el coste es diferido y solo lo asumen quienes finalmente vuelven a comprar.


In [51]:
# ── Análisis de rentabilidad del Pack 5 revisiones + bono 1.000€ ────────────
TASA_REDENCION = {'Pesimista': 0.10, 'Base': 0.20, 'Optimista': 0.30}

clientes_pack = clientes_cltv[clientes_cltv['segmento'].isin(['BAJO', 'MUY_BAJO'])].copy()

def beneficio_pack_fn(row):
    base        = row['Mantenimiento_medio']
    alpha       = 0.07 if row['Modelo'] in ['A', 'B'] else 0.10
    n0          = row['Revisiones']
    ingreso_5   = sum(base * ((1 + alpha) ** (n0 + k)) for k in range(1, 6))
    precio_pack = ingreso_5 * 0.90
    return precio_pack * 0.62

clientes_pack['beneficio_pack'] = clientes_pack.apply(beneficio_pack_fn, axis=1)

RR_PACK = {'BAJO': 0.35, 'MUY_BAJO': 0.30}

resultados_pack = []
for seg in ['BAJO', 'MUY_BAJO']:
    g     = clientes_pack[clientes_pack['segmento'] == seg]
    n_acc = len(g) * RR_PACK[seg]
    benef_pack_total = n_acc * g['beneficio_pack'].mean()
    for escenario, tasa in TASA_REDENCION.items():
        coste_bono = n_acc * tasa * 1000
        benef_neto = benef_pack_total - coste_bono
        resultados_pack.append({
            'Segmento'       : seg,
            'Escenario'      : escenario,
            'N clientes'     : len(g),
            'N aceptan pack' : int(n_acc),
            'N redimen bono' : int(n_acc * tasa),
            'Benef. pack (€)': round(benef_pack_total),
            'Coste bono (€)' : round(coste_bono),
            'Benef. neto (€)': round(benef_neto),
            'ROI pack'       : round(benef_neto / coste_bono, 2) if coste_bono > 0 else float('inf'),
        })

df_pack_roi = pd.DataFrame(resultados_pack)
display(df_pack_roi)

# ── Impacto sobre el ROI global (escenario Base) ─────────
coste_bono_base_total = df_pack_roi[df_pack_roi['Escenario'] == 'Base']['Coste bono (€)'].sum()
roi_ajustado = (beneficio_total_cltv - coste_bono_base_total) / (costes_total_cltv + coste_bono_base_total)

print(f'ROI Churn×CLTV sin bono:     {roi_global_cltv:.4f}x')
print(f'Coste total del bono 1.000€: {coste_bono_base_total:,.0f} €')
print(f'ROI Churn×CLTV con bono:     {roi_ajustado:.4f}x  ({roi_ajustado - roi_global_cltv:+.4f})')
print('→ El bono reduce el ROI ligeramente, pero asegura fidelidad a largo plazo y una 2ª venta.')

,Segmento,Escenario,N clientes,N aceptan pack,N redimen bono,Benef. pack (€),Coste bono (€),Benef. neto (€),ROI pack
0,BAJO,Pesimista,31,10,1,14636,1085,13551,12.49
1,BAJO,Base,31,10,2,14636,2170,12466,5.74
2,BAJO,Optimista,31,10,3,14636,3255,11381,3.50
3,MUY_BAJO,Pesimista,1450,435,43,490046,43500,446546,10.27
4,MUY_BAJO,Base,1450,435,87,490046,87000,403046,4.63
5,MUY_BAJO,Optimista,1450,435,130,490046,130500,359546,2.76


ROI Churn×CLTV sin bono:     2.7510x
Coste total del bono 1.000€: 89,170 €
ROI Churn×CLTV con bono:     1.8619x  (-0.8891)
→ El bono reduce el ROI ligeramente, pero asegura fidelidad a largo plazo y una 2ª venta.


---

## 10. Resumen Ejecutivo

Síntesis de los resultados del plan comercial completo: cartera analizada, comparativa de estrategias y decisiones clave de negocio.

In [52]:
# ── Cartera ─────────────────────────────────────────────────
print(f'Clientes analizados:  {len(clientes_cltv):,}')
print(f'CLTV medio:           {clientes_cltv["CLTV"].mean():,.0f} €')
print(f'CLTV total cartera:   {clientes_cltv["CLTV"].sum():,.0f} €')

# ── Comparativa de estrategias ───────────────────────────────
ahorro = costes_total - costes_total_cltv
extra  = beneficio_total_cltv - beneficio_total

df_resumen = pd.DataFrame({
    'Métrica': ['Inversión total (€)', 'Beneficio neto (€)', 'ROI global', 'Clientes que responden'],
    'Solo-Churn': [
        f'{costes_total:,.0f}', f'{beneficio_total:,.0f}',
        f'{roi_global:.2f}x', f'{int(df_roi["N_Responden"].sum()):,}',
    ],
    'Churn × CLTV': [
        f'{costes_total_cltv:,.0f}', f'{beneficio_total_cltv:,.0f}',
        f'{roi_global_cltv:.2f}x', f'{int(df_roi_cltv["N_Resp"].sum()):,}',
    ],
    'Diferencia': [
        f'{costes_total_cltv - costes_total:+,.0f}', f'{beneficio_total_cltv - beneficio_total:+,.0f}',
        f'{roi_global_cltv - roi_global:+.2f}', f'{int(df_roi_cltv["N_Resp"].sum()) - int(df_roi["N_Responden"].sum()):+,}',
    ],
})
display(df_resumen)

# ── Decisiones clave ─────────────────────────────────────────
print('Decisiones clave:')
print('  1. Concentrar inversión en ALTO riesgo + ALTO valor → máximo ROI')
print('  2. No gastar en clientes de BAJO valor → coste no justificado')
print('  3. Pack 5 rev. + bono 1.000€ para bajo riesgo → fidelidad a largo plazo')
print('  4. Monitorizar segmento MEDIO → escalar acción si sube el riesgo')
print('  5. Activar bono 2º vehículo cuando el cliente alcance n≥5 revisiones')
print(f'\nRecomendación: adoptar Churn×CLTV — ahorra {ahorro:,.0f} € y genera {extra:,.0f} € adicionales.')

Clientes analizados:  10,000
CLTV medio:           850 €
CLTV total cartera:   8,496,291 €


,Métrica,Solo-Churn,Churn × CLTV,Diferencia
0,Inversión total (€),"492,521","287,038","-205,483"
1,Beneficio neto (€),"762,562","789,647","+27,085"
2,ROI global,1.55x,2.75x,+1.20
3,Clientes que responden,"6,077","5,108",-969


Decisiones clave:
  1. Concentrar inversión en ALTO riesgo + ALTO valor → máximo ROI
  2. No gastar en clientes de BAJO valor → coste no justificado
  3. Pack 5 rev. + bono 1.000€ para bajo riesgo → fidelidad a largo plazo
  4. Monitorizar segmento MEDIO → escalar acción si sube el riesgo
  5. Activar bono 2º vehículo cuando el cliente alcance n≥5 revisiones

Recomendación: adoptar Churn×CLTV — ahorra 205,483 € y genera 27,085 € adicionales.
